# Generating Music with Machine Learning
**UE24CS352A – ML Mini-Project (Problem 22)**

Pipeline: MIDI → note-event sequences → models (Markov baseline, LSTM, GRU) → evaluation → generated `.mid` files.

Run top to bottom on Colab (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip -q install pretty_midi

In [ ]:
import os, glob, random, math, collections, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pretty_midi, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

GRID = 0.125        # time quantisation (seconds)
MAX_IOI = 16        # max gap to next onset, in grid units (= 2 s)
MAX_CHORD = 4       # keep at most the 4 highest notes per onset
SEQ_LEN = 64        # events per training window
VOCAB_SIZE = 1500   # most frequent events kept, rest -> <unk>
MAX_FILES = 120     # number of MIDI files to use
EPOCHS = 30
print('device:', DEVICE)

## 1. Dataset
MAESTRO v3 (piano MIDI). We use Chopin pieces, as in the reference paper (Chopin etudes).
To use your own files instead, upload them to a folder called `midi_files/` and skip the download.

In [ ]:
files = sorted(glob.glob('midi_files/**/*.mid*', recursive=True))
if not files:
    BASE = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/'
    if not os.path.exists('maestro-v3.0.0'):
        urllib.request.urlretrieve(BASE + 'maestro-v3.0.0-midi.zip', 'maestro_midi.zip')
        zipfile.ZipFile('maestro_midi.zip').extractall('.')
    meta = pd.read_csv('maestro-v3.0.0/maestro-v3.0.0.csv')
    meta = meta[meta.canonical_composer.str.contains('Chopin')]
    files = ['maestro-v3.0.0/' + f for f in meta.midi_filename][:MAX_FILES]
print(len(files), 'MIDI files')

## 2. Preprocessing
Each song becomes a sequence of **events**. An event = (set of pitches starting at the same time, gap to the next onset).
Splitting is done **per song** (80/10/10) so no windows from the same piece leak between train and test.

In [ ]:
def midi_to_events(path):
    try:
        pm = pretty_midi.PrettyMIDI(path)
    except Exception:
        return []
    onsets = collections.defaultdict(set)
    for inst in pm.instruments:
        if inst.is_drum:
            continue
        for n in inst.notes:
            onsets[int(round(n.start / GRID))].add(n.pitch)
    keys = sorted(onsets)
    ev = []
    for i, k in enumerate(keys):
        nxt = keys[i + 1] if i + 1 < len(keys) else k + 4
        ioi = int(min(max(nxt - k, 1), MAX_IOI))
        pitches = tuple(sorted(onsets[k])[-MAX_CHORD:])
        ev.append((pitches, ioi))
    return ev

random.shuffle(files)
n = len(files)
split_files = dict(train=files[:int(.8 * n)], val=files[int(.8 * n):int(.9 * n)], test=files[int(.9 * n):])
songs = {k: [s for s in (midi_to_events(f) for f in v) if len(s) > SEQ_LEN + 2] for k, v in split_files.items()}
for k, v in songs.items():
    print(k, len(v), 'songs,', sum(map(len, v)), 'events')

# vocabulary built on TRAIN only
cnt = collections.Counter(e for s in songs['train'] for e in s)
itos = ['<unk>'] + [e for e, _ in cnt.most_common(VOCAB_SIZE - 1)]
stoi = {e: i for i, e in enumerate(itos)}
data = {k: [[stoi.get(e, 0) for e in s] for s in v] for k, v in songs.items()}
print('vocab size:', len(itos), '| unk rate (test):',
      round(np.mean([t == 0 for s in data['test'] for t in s]), 3))

class Windows(Dataset):
    def __init__(self, seqs, L=SEQ_LEN, stride=4):
        xs, ys = [], []
        for q in seqs:
            for i in range(0, len(q) - L - 1, stride):
                xs.append(q[i:i + L]); ys.append(q[i + 1:i + L + 1])
        self.x, self.y = torch.tensor(xs), torch.tensor(ys)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

train_dl = DataLoader(Windows(data['train'], stride=4), batch_size=128, shuffle=True)
val_dl = DataLoader(Windows(data['val'], stride=32), batch_size=256)
test_dl = DataLoader(Windows(data['test'], stride=32), batch_size=256)
print('windows:', len(train_dl.dataset), len(val_dl.dataset), len(test_dl.dataset))

## 3. Baseline: Markov chain (bigram)
Predicts the next event from the previous one using counts from the training set (same spirit as the paper's Naive-Bayes-like model).

In [ ]:
bi = collections.defaultdict(collections.Counter)
for q in data['train']:
    for a, b in zip(q[:-1], q[1:]):
        bi[a][b] += 1
glob_top = [t for t, _ in collections.Counter(t for q in data['train'] for t in q).most_common(5)]
top5 = {a: [t for t, _ in c.most_common(5)] for a, c in bi.items()}

def markov_eval(seqs):
    h1 = h5 = tot = 0
    for q in seqs:
        for a, b in zip(q[:-1], q[1:]):
            if b == 0:
                continue
            p = top5.get(a, glob_top)
            tot += 1; h1 += (p[0] == b); h5 += (b in p)
    return h1 / tot, h5 / tot

mk_acc, mk_top5 = markov_eval(data['test'])
print(f'Markov baseline  test acc={mk_acc:.3f}  top5={mk_top5:.3f}')

## 4. Neural models: LSTM and GRU
Embedding → 2-layer recurrent net (dropout 0.3) → softmax over the event vocabulary. Trained with cross-entropy, Adam, gradient clipping, and early stopping on validation loss.

In [ ]:
class MusicRNN(nn.Module):
    def __init__(self, V, kind='lstm', emb=128, hid=256, layers=2, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        R = nn.LSTM if kind == 'lstm' else nn.GRU
        self.rnn = R(emb, hid, layers, batch_first=True, dropout=drop)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hid, V)
    def forward(self, x, h=None):
        o, h = self.rnn(self.drop(self.emb(x)), h)
        return self.fc(self.drop(o)), h

crit = nn.CrossEntropyLoss(ignore_index=0)   # ignore <unk> targets

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    tl = c1 = c5 = tot = batches = 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out, _ = model(x)
        tl += crit(out.reshape(-1, out.size(-1)), y.reshape(-1)).item(); batches += 1
        mask = y != 0
        t5 = out.topk(5, dim=-1).indices
        c1 += ((t5[..., 0] == y) & mask).sum().item()
        c5 += ((t5 == y.unsqueeze(-1)).any(-1) & mask).sum().item()
        tot += mask.sum().item()
    return tl / batches, c1 / tot, c5 / tot

def run(kind, epochs=EPOCHS, patience=4, lr=2e-3):
    m = MusicRNN(len(itos), kind).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    best, bad, hist = 1e9, 0, {'train': [], 'val': []}
    for ep in range(epochs):
        m.train(); tl = nb = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out, _ = m(x)
            loss = crit(out.reshape(-1, out.size(-1)), y.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            tl += loss.item(); nb += 1
        vl, va, _ = evaluate(m, val_dl)
        hist['train'].append(tl / nb); hist['val'].append(vl)
        print(f'[{kind}] epoch {ep+1:02d} train {tl/nb:.3f} val {vl:.3f} val_acc {va:.3f}')
        if vl < best:
            best, bad = vl, 0
            torch.save(m.state_dict(), f'{kind}.pt')
        else:
            bad += 1
            if bad >= patience:
                print('early stopping'); break
    m.load_state_dict(torch.load(f'{kind}.pt'))
    return m, hist

lstm, h_lstm = run('lstm')
gru, h_gru = run('gru')

## 5. Results

In [ ]:
rows = [dict(model='Markov (bigram)', test_loss=np.nan, perplexity=np.nan, top1=mk_acc, top5=mk_top5)]
for name, m in [('LSTM', lstm), ('GRU', gru)]:
    l, a1, a5 = evaluate(m, test_dl)
    rows.append(dict(model=name, test_loss=l, perplexity=math.exp(l), top1=a1, top5=a5))
results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
results.to_csv('results.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, (nm, h) in zip(ax, [('LSTM', h_lstm), ('GRU', h_gru)]):
    a.plot(h['train'], label='train'); a.plot(h['val'], label='val')
    a.set_title(f'{nm} loss'); a.set_xlabel('epoch'); a.set_ylabel('cross-entropy'); a.legend()
plt.tight_layout(); plt.savefig('loss_curves.png', dpi=150); plt.show()

## 6. Generate music
Seed with the first 32 events of a held-out test song, then sample (temperature + top-k). Outputs are written as `.mid` files.

In [ ]:
def tokens_to_midi(tokens, path):
    pm = pretty_midi.PrettyMIDI(); inst = pretty_midi.Instrument(0); t = 0.0
    for tk in tokens:
        if tk == 0:
            continue
        pitches, ioi = itos[tk]; d = ioi * GRID
        for p in pitches:
            inst.notes.append(pretty_midi.Note(80, int(p), t, t + max(d, 0.2)))
        t += d
    pm.instruments.append(inst); pm.write(path); return pm

@torch.no_grad()
def generate(model, seed, n=300, temp=0.9, topk=15):
    model.eval()
    out = list(seed)
    logits, h = model(torch.tensor([seed], device=DEVICE))
    for _ in range(n):
        l = logits[0, -1] / temp
        l[0] = -float('inf')                       # never sample <unk>
        v, idx = l.topk(topk)
        nxt = idx[torch.multinomial(torch.softmax(v, -1), 1)].item()
        out.append(nxt)
        logits, h = model(torch.tensor([[nxt]], device=DEVICE), h)
    return out

def markov_generate(seed, n=300):
    out = list(seed)
    for _ in range(n):
        c = bi.get(out[-1])
        if not c:
            out.append(random.choice(range(1, len(itos)))); continue
        toks, w = zip(*[(t, k) for t, k in c.items() if t != 0] or [(1, 1)])
        out.append(random.choices(toks, weights=w)[0])
    return out

seed = data['test'][0][:32]
os.makedirs('generated', exist_ok=True)
tokens_to_midi(data['test'][0][:332], 'generated/0_original.mid')
tokens_to_midi(markov_generate(seed), 'generated/1_markov.mid')
tokens_to_midi(generate(lstm, seed), 'generated/2_lstm.mid')
tokens_to_midi(generate(gru, seed), 'generated/3_gru.mid')
print(os.listdir('generated'))

## 7. Listen in the notebook (optional) + download

In [ ]:
!apt-get -qq install -y fluidsynth > /dev/null
!pip -q install pyfluidsynth
import importlib, sys
# pretty_midi checks for pyfluidsynth when it is first imported, so reload it now that it is installed
importlib.reload(importlib.import_module('pretty_midi.fluidsynth'))
from IPython.display import Audio, display
for f in sorted(os.listdir('generated')):
    print(f)
    pm = pretty_midi.PrettyMIDI('generated/' + f)
    display(Audio(pm.fluidsynth(fs=22050), rate=22050))

In [ ]:
!zip -qr generated.zip generated results.csv loss_curves.png
from google.colab import files
files.download('generated.zip')